# Colab Fine-Tuning: MiniLM Encoder (Person 3)

This notebook demonstrates fine-tuning `sentence-transformers/all-MiniLM-L6-v2` on recipe-query pairs using `MultipleNegativesRankingLoss`, generating a validated serving bundle (`minilm_finetuned`), tracking the run in MLflow, and inspecting monitoring metrics.

> **Recommended Runtime:** GPU (`Runtime > Change runtime type > T4 GPU`).

## 1. Setup & Dependencies

In [ ]:
# Clone repository if running on Colab
import os, subprocess
if not os.path.exists("/content/repo"):
    subprocess.run(["git", "clone", "https://github.com/minhhoangb/food-recipe-recommendation-system.git", "/content/repo"], check=True)
    os.chdir("/content/repo")

!pip install -q -r requirements.txt
!python scripts/verify_data.py

## 2. Build Training Pairs

Constructs query-positive pairs and hard negatives from `data/processed/pilot/` using `src.build_pairs`.

In [ ]:
!python -m src.build_pairs --config configs/pilot.yaml --split train --mode ingredients --negatives

## 3. End-to-End Fine-Tuning & Serving Bundle Export

Executes `src.train` to:
1. Fine-tune the MiniLM encoder with MultipleNegativesRankingLoss.
2. Encode the full pilot corpus and create a valid bundle under `models/minilm_finetuned-...`.
3. Evaluate retrieval performance on the validation split with `ranx`.
4. Log parameters, metrics, and bundle artifacts to MLflow, and register the model with alias `staging`.

In [ ]:
!python -m src.train --config configs/pilot.yaml --epochs 2 --batch-size 32 --mode ingredients

## 4. Evaluate Fine-Tuned Retriever vs Pretrained

Compare the resulting fine-tuned retriever with the baseline pretrained MiniLM encoder.

In [ ]:
!python -m src.evaluation --mode ingredients --config configs/pilot.yaml --split validation

## 5. API Request & Model Monitoring

Inspect request logs, error rates, p95 latencies, and ingredient distribution drift using `src.monitor`.

In [ ]:
!python -m src.monitor --log-path logs/requests.jsonl --out reports/monitoring_report.md